
# 01 — Préparation des données Silver

## Objectif
Transformer les observations météorologiques brutes de Météo-France en données structurées, contrôlées et exploitables pour les analyses et les visualisations.

## Traitements réalisés
- Lecture des données Bronze : `dbo.raw_meteo_france`.
- Extraction des champs JSON : `dbo.raw_meteo_france_parsing`.
- Contrôle des valeurs manquantes.
- Conversion des températures de Kelvin en Celsius.
- Vérification des coordonnées géographiques et de l'humidité.
- Enregistrement des données dans `dbo.silver_meteo_france`.

## Automatisation
Ce notebook sera exécuté quotidiennement par le pipeline `PL_Meteo_Transformation_journalière`, après la copie des données depuis Eventhouse.

## Résultat attendu
Une table Silver fiable et actualisée quotidiennement, destinée aux analyses météorologiques et aux futurs croisements avec les données OpenRadiation.


In [8]:

# ==================================================
# ÉTAPE 01.1 — LECTURE DES DONNÉES BRONZE
# ==================================================

# A — Lire les observations copiées depuis Eventhouse
df_raw = spark.table("dbo.raw_meteo_france")

# B — Vérifier le nombre d'observations
print("Nombre de relevés Bronze :", df_raw.count())

# C — Vérifier la structure des données
df_raw.printSchema()

# D — Afficher les 5 premières observations
display(df_raw.limit(5))


StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 10, Finished, Available, Finished, False)

Nombre de relevés Bronze : 127130
root
 |-- source: string (nullable = true)
 |-- observation_date: timestamp (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- data: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 011559d9-6019-4f2d-aa68-4c80c1a67c61)


## 02 — Parsing des données JSON

**Objectif :** extraire les informations météorologiques du JSON brut.

**Source :** dbo.raw_meteo_france

**Destination :** dbo.raw_meteo_france_parsing

**Traitement :** extraction des coordonnées, températures, humidité et autres mesures, sans modifier les données brutes.


In [2]:
# ÉTAPE 2 — Exploration de la structure JSON

from pyspark.sql import functions as F

# Afficher un exemple complet de JSON
exemple_json = (
    df_raw
    .where(F.col("data").isNotNull())
    .select("data")
    .first()
)

print(exemple_json["data"] if exemple_json else "Aucun JSON disponible")

StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 4, Finished, Available, Finished, False)

{
  "lat": 46.278167,
  "lon": 5.669,
  "geo_id_insee": "01014002",
  "reference_time": "2026-11-06T08:09:06.0000000Z",
  "insert_time": "2026-11-06T08:07:39.0000000Z",
  "validity_time": "2026-11-06T08:06:00.0000000Z",
  "t": 279.15,
  "td": 279.15,
  "u": 100.0,
  "dd": 360.0,
  "ff": 0.6,
  "ddraf10": 350.0,
  "raf10": 1.0,
  "rr_per": 0.0,
  "t_10": null,
  "t_20": null,
  "t_50": null,
  "t_100": null,
  "vv": null,
  "etat_sol": null,
  "sss": 0.0,
  "insolh": null,
  "ray_glo01": null,
  "pres": null,
  "pmer": null
}


In [3]:

# ============================================================
# ÉTAPE 2-1 — PARSING JSON DES OBSERVATIONS MÉTÉO-FRANCE
# ============================================================

from pyspark.sql import functions as F

# A — Lire les données brutes
df_raw = spark.table("dbo.raw_meteo_france")

# B — Extraire les champs du JSON
df_parsing = (
    df_raw
    .withColumn("latitude", F.get_json_object("data", "$.lat").cast("double"))
    .withColumn("longitude", F.get_json_object("data", "$.lon").cast("double"))
    .withColumn("geo_id_insee", F.get_json_object("data", "$.geo_id_insee"))
    .withColumn("temperature_k", F.get_json_object("data", "$.t").cast("double"))
    .withColumn("humidite_pct", F.get_json_object("data", "$.u").cast("double"))
    .withColumn("vitesse_vent", F.get_json_object("data", "$.ff").cast("double"))
    .withColumn("direction_vent", F.get_json_object("data", "$.dd").cast("double"))
    .withColumn("precipitations", F.get_json_object("data", "$.rr_per").cast("double"))
)

# C — Vérifier les résultats
print("Nombre de relevés après parsing :", df_parsing.count())

display(
    df_parsing.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_k",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(10)
)


StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 5, Finished, Available, Finished, False)

Nombre de relevés après parsing : 127130


SynapseWidget(Synapse.DataFrame, 72a5218b-860f-4c0e-b517-6ebe68b84d84)

In [4]:
# ==================================================
# ÉTAPE 02.2 — ENREGISTREMENT DU PARSING JSON
# ==================================================

# Enregistrer les données parsées dans le Lakehouse
(
    df_parsing.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbo.raw_meteo_france_parsing")
)

# Vérifier l'enregistrement
df_verification = spark.table("dbo.raw_meteo_france_parsing")

print("Nombre de relevés enregistrés :", df_verification.count())

display(df_verification.select(
    "observation_date",
    "latitude",
    "longitude",
    "temperature_k",
    "humidite_pct"
).limit(5))

StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 6, Finished, Available, Finished, False)

Nombre de relevés enregistrés : 127130


SynapseWidget(Synapse.DataFrame, 48e69b7d-227d-48df-9ba2-d4174d520b8b)


## 03 — Nettoyage des données Silver

**Objectif :** nettoyer et fiabiliser les observations météorologiques.

**Source :** dbo.raw_meteo_france_parsing

**Destination :** dbo.silver_meteo_france

**Traitements :** contrôle des valeurs manquantes, conversion des températures en °C et vérification des coordonnées géographiques.


In [5]:

# ==================================================
# ÉTAPE 03.1 — CONTRÔLE QUALITÉ AVANT NETTOYAGE
# ==================================================

from pyspark.sql import functions as F

# A — Lire les données après parsing JSON
df_parsing = spark.table("dbo.raw_meteo_france_parsing")

# B — Sélectionner les colonnes à contrôler
colonnes = [
    "latitude",
    "longitude",
    "temperature_k",
    "humidite_pct",
    "vitesse_vent"
]

# C — Compter les valeurs manquantes
controle_qualite = df_parsing.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in colonnes
])

# D — Afficher les résultats
print("Nombre total de relevés :", df_parsing.count())

display(controle_qualite)


StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 7, Finished, Available, Finished, False)

Nombre total de relevés : 127130


SynapseWidget(Synapse.DataFrame, 62630b28-c126-41ac-8343-2c49b1c8b2cf)

In [6]:

# ==================================================
# ÉTAPE 03.2 — NETTOYAGE DES DONNÉES SILVER
# ==================================================

from pyspark.sql import functions as F

# A — Lire les données après parsing
df_parsing = spark.table("dbo.raw_meteo_france_parsing")

# B — Convertir la température en degrés Celsius
df_silver = df_parsing.withColumn(
    "temperature_c",
    F.round(F.col("temperature_k") - 273.15, 2)
)

# C — Vérifier les coordonnées géographiques
df_silver = df_silver.filter(
    F.col("latitude").between(-90, 90) &
    F.col("longitude").between(-180, 180)
)

# D — Corriger les valeurs d'humidité invalides
# Les valeurs manquantes restent NULL
df_silver = df_silver.withColumn(
    "humidite_pct",
    F.when(
        F.col("humidite_pct").between(0, 100),
        F.col("humidite_pct")
    ).otherwise(F.lit(None).cast("double"))
)

# E — Afficher les résultats
print("Nombre de relevés Silver :", df_silver.count())

display(
    df_silver.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_c",
        "humidite_pct",
        "vitesse_vent"
    ).limit(10)
)


StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 8, Finished, Available, Finished, False)

Nombre de relevés Silver : 127130


SynapseWidget(Synapse.DataFrame, 971f7fae-e1b8-4eb5-8313-15adc4f8b86f)

In [7]:

# ==================================================
# ÉTAPE 03.3 — ENREGISTREMENT DE LA TABLE SILVER
# ==================================================

# A — Enregistrer les données nettoyées
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbo.silver_meteo_france")

# B — Vérifier la création de la table
df_verification = spark.table("dbo.silver_meteo_france")

print("Nombre de relevés enregistrés :", df_verification.count())

# C — Afficher un aperçu
display(
    df_verification.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_c",
        "humidite_pct",
        "vitesse_vent"
    ).limit(10)
)


StatementMeta(, 3371adf8-5dfb-49bf-9be3-14894995ae84, 9, Finished, Available, Finished, False)

Nombre de relevés enregistrés : 127130


SynapseWidget(Synapse.DataFrame, 59ee6322-9e82-4f70-9aa9-c4c5570e7141)